In [1]:
# Necessary imports
import re
import emoji
from gtrans import translate_text, translate_html
import random
import pandas as pd
import numpy as np
from multiprocessing import  Pool
import time

In [2]:
# Function to remove emojis in text, since these conflict during translation
def remove_emoji(text):
    return emoji.get_emoji_regexp().sub(u'', text)


def approximate_emoji_insert(string, index,char):
    if(index<(len(string)-1)):
        
        while(string[index]!=' ' ):
            if(index+1==len(string)):
                break
            index=index+1
        return string[:index] + ' '+char + ' ' + string[index:]
    else:
        return string + ' '+char + ' ' 
    


def extract_emojis(str1):
    try:
        return [(c,i) for i,c in enumerate(str1) if c in emoji.UNICODE_EMOJI]
    except AttributeError:
        return []

In [3]:
# Use multiprocessing framework for speeding up translation process
def parallelize_dataframe(df, func, n_cores=4):
    '''parallelize the dataframe'''
    df_split = np.array_split(df, n_cores)
    pool = Pool(n_cores)
    df = pd.concat(pool.map(func, df_split))
    pool.close()
    pool.join()
    return df

# Main function for translation
def translate(x,lang):
    '''provide the translation given text and the language'''
    #x=preprocess_lib.preprocess_multi(x,lang,multiple_sentences=False,stop_word_remove=False, tokenize_word=False, tokenize_sentence=False)
    emoji_list=extract_emojis(x)
    try:
        translated_text=translate_text(x,lang,'en')
    except:
        translated_text=x
    for ele in emoji_list:
        translated_text=approximate_emoji_insert(translated_text, ele[1],ele[0])
    return translated_text

def add_features(df):
    '''adding new features to the dataframe'''
    translated_text=[]
    for index,row in df.iterrows():
        if(row['lang']in ['en','unk']):
            translated_text.append(row['text'])
        else:
            translated_text.append(translate(row['text'],row['lang']))    
    df["translated"]=translated_text
    return df

In [4]:
import glob 
file_path = 'full_data'
files = glob.glob(file_path+'/*.csv')

In [ ]:
import os
from tqdm import tqdm

size = 50  # Process 50 rows at a time
n_cores = 4  # Reduced from 20 to avoid overhead

for file in files:
    print(f"\nProcessing {os.path.basename(file)}...")
    wp_data = pd.read_csv(file)
    print(f"Total rows: {len(wp_data)}")
    
    list_df = []
    total_rows = len(wp_data)
    
    # Process all rows, not just first 100
    for i in tqdm(range(0, total_rows, size)):
        end_idx = min(i + size, total_rows)
        print(f"Processing rows {i} to {end_idx}...")
        try:
            df_new = parallelize_dataframe(wp_data[i:end_idx], add_features, n_cores=n_cores)
            list_df.append(df_new)
        except Exception as e:
            print(f"Error processing rows {i}-{end_idx}: {e}")
            # If translation fails, just use original data
            list_df.append(wp_data[i:end_idx])
    
    df_translated = pd.concat(list_df, ignore_index=True)
    
    base = os.path.basename(file)
    root, ext = os.path.splitext(base)
    file_name = f"{root}_translated{ext}"
    
    output_path = os.path.join(file_path, file_name)
    df_translated.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")

print("\nAll files translated successfully!")



Processing Arabic_1b_full.csv...
Total rows: 3353


  0%|          | 0/68 [00:00<?, ?it/s]

Processing rows 0 to 50...


c:\Users\etulyon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
